### 비용 최적화 시나리오 계산
- 파일 : nA5_report_calc.ipynb
- 역할 : 미니프로젝트 예시 시나리오의 사용량·비용·용량을 계산기로 확인합니다.
- 기능 :
    1. 시나리오 사용량과 비용 계산  


# 5장 미니프로젝트: 예시 시나리오 계산 (4장 계산기에 이어서)

부록 원고의 코드와 동일합니다. 핵심 수식은 코드의 주석에 함께 적어 두었습니다.

In [ ]:
#- 비용 계산기 노트북(nA4)의 정의를 그대로 재사용 (nA4를 먼저 실행했다면 건너뜀)
PRICE = {
    "mid":   {"in": 2.00, "out": 12.00},
    "light": {"in": 0.20, "out": 1.20},
}
# 클라우드 월 비용 (비용 계산기와 동일)
def cloud_monthly(req_per_day, tok_in, tok_out, price, days=30):
    m_in  = req_per_day * days * tok_in  / 1e6
    m_out = req_per_day * days * tok_out / 1e6
    return m_in * price["in"] + m_out * price["out"]
# 온디바이스 월 비용 (상각 + 전력)
def ondevice_monthly(device_usd, life_months, watts,
                     krw_per_kwh=160, krw_per_usd=1400, days=30):
    amort = device_usd / life_months
    kwh   = watts * 24 * days / 1000
    elec  = kwh * krw_per_kwh / krw_per_usd
    return amort + elec, amort, elec

# 확인 포인트 ------------------------------------------
# - 함수 정의 셀: 출력 없음

>> [코드 A5-1] 시나리오 사용량과 비용 계산 (nA4_cost_calc.ipynb에 이어서)
<hr>

In [ ]:
# 비용 계산기 노트북의 PRICE, cloud_monthly, ondevice_monthly를 그대로 사용

#- 미니프로젝트 예시 시나리오: 하루 400건, 입력 1,200·출력 250토큰
req_day, tok_in, tok_out = 400, 1200, 250   # 200명 × 2건

#- 클라우드와 온디바이스 월 비용
for name in PRICE:
    print(f"{name:5s} 모델: 월 {cloud_monthly(req_day, tok_in, tok_out, PRICE[name]):.1f}달러")
total, amort, elec = ondevice_monthly(device_usd=210, life_months=36, watts=1.87)
print(f"온디바이스: 월 {total:.2f}달러")

#- 손익분기 환산
mix = (tok_in * PRICE["mid"]["in"] + tok_out * PRICE["mid"]["out"]) / (tok_in + tok_out)
be_req_day = total / mix * 1e6 / (tok_in + tok_out) / 30
print(f"중간급 모델 대비 손익분기: 하루 {be_req_day:.0f}건")

#- 용량 점검: 월 필요량과 피크일
need = req_day * 30 * tok_out / 1e6
peak_day = req_day * 3 * tok_out / 1e6
print(f"월 출력 필요량 {need:.1f}M (장비 상한 17.9M), 피크일 {peak_day:.1f}M (일 상한 0.60M)")

# 확인 포인트 --------------------------------------------------------------------------
# - 출력 결과 체크

**실행 결과 예시**
```
mid   모델: 월 64.8달러
light 모델: 월 6.5달러
온디바이스: 월 5.99달러
중간급 모델 대비 손익분기: 하루 37건
월 출력 필요량 3.0M (장비 상한 17.9M), 피크일 0.3M (일 상한 0.60M)
```